# Star Wars Survey

While waiting for *Star Wars: The Force Awakens* to come out, the team at FiveThirtyEight surveyed Star Wars fans using SurveyMonkey. The cleaned dataset we use contains 1,186 responses. One of the questions they wanted to answer: does the rest of America realize that *The Empire Strikes Back* is clearly the best of the bunch?

In this project, we clean the survey data and explore it to find:

- which movie is ranked the highest,
- which movie has been seen by the most respondents,
- how the answers differ between segments of respondents (gender, fans vs. non-fans).

## Reading the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

star_wars = pd.read_csv("star_wars.csv", encoding="ISO-8859-1")
star_wars = star_wars[star_wars["RespondentID"].notnull()]
print(star_wars.shape)
star_wars.head(10)

In [ ]:
star_wars.columns

The dataset has 1,186 rows and 38 columns. `RespondentID` is unique for every respondent; as a safety check we keep only the rows where it isn't missing. Several questions were answered with checkboxes, which is why some columns have names like `Unnamed: 4` — these need cleaning.

## Cleaning and Mapping Yes/No Columns

In [ ]:
yes_no = {"Yes": True, "No": False}

for col in ["Have you seen any of the 6 films in the Star Wars franchise?",
            "Do you consider yourself to be a fan of the Star Wars film franchise?"]:
    star_wars[col] = star_wars[col].map(yes_no)
    print(star_wars[col].value_counts(dropna=False), "\n")

## Cleaning and Mapping Checkbox Columns

Columns 3 to 8 represent a single checkbox question: *Which of the following Star Wars films have you seen?* Each column holds the name of a movie if the respondent saw it, or `NaN` otherwise. We convert each column to a Boolean and give it a clearer name (`seen_1` for Episode I, ..., `seen_6` for Episode VI).

In [ ]:
movie_mapping = {
    "Star Wars: Episode I  The Phantom Menace": True,
    "Star Wars: Episode II  Attack of the Clones": True,
    "Star Wars: Episode III  Revenge of the Sith": True,
    "Star Wars: Episode IV  A New Hope": True,
    "Star Wars: Episode V The Empire Strikes Back": True,
    "Star Wars: Episode VI Return of the Jedi": True,
    np.nan: False
}

for col in star_wars.columns[3:9]:
    star_wars[col] = star_wars[col].map(movie_mapping)

star_wars = star_wars.rename(columns={
    "Which of the following Star Wars films have you seen? Please select all that apply.": "seen_1",
    "Unnamed: 4": "seen_2",
    "Unnamed: 5": "seen_3",
    "Unnamed: 6": "seen_4",
    "Unnamed: 7": "seen_5",
    "Unnamed: 8": "seen_6"
})
star_wars.iloc[:, 3:9].head()

## Cleaning the Ranking Columns

Columns 9 to 14 ask respondents to rank the six movies from 1 (favorite) to 6 (least favorite). We convert them to numeric values and rename them `ranking_1` to `ranking_6`.

In [ ]:
star_wars[star_wars.columns[9:15]] = star_wars[star_wars.columns[9:15]].astype(float)

star_wars = star_wars.rename(columns={
    "Please rank the Star Wars films in order of preference with 1 being your favorite film in the franchise and 6 being your least favorite film.": "ranking_1",
    "Unnamed: 10": "ranking_2",
    "Unnamed: 11": "ranking_3",
    "Unnamed: 12": "ranking_4",
    "Unnamed: 13": "ranking_5",
    "Unnamed: 14": "ranking_6"
})
star_wars.iloc[:, 9:15].head()

## Finding the Highest-Ranked Movie

In [ ]:
ranking_cols = ["ranking_%d" % i for i in range(1, 7)]
seen_cols = ["seen_%d" % i for i in range(1, 7)]
episodes = ["I", "II", "III", "IV", "V", "VI"]

mean_ranking = star_wars[ranking_cols].mean()
print(mean_ranking)

plt.bar(episodes, mean_ranking)
plt.title("Average ranking per movie (lower is better)")
plt.xlabel("Episode")
plt.ylabel("Average ranking")
plt.show()

Remember that a **lower** average ranking is better (1 = favorite).

- **Episode V – The Empire Strikes Back** has the best average ranking (**2.51**), followed by Episode VI – Return of the Jedi (3.05) and Episode IV – A New Hope (3.27).
- The prequels rank lowest: Episode I (3.73), Episode II (4.09) and Episode III (4.34).

So respondents clearly prefer the original trilogy, and FiveThirtyEight's hunch is confirmed: *The Empire Strikes Back* is the favorite movie.

## Finding the Most Viewed Movie

In [ ]:
seen_counts = star_wars[seen_cols].sum()
print(seen_counts)

plt.bar(episodes, seen_counts)
plt.title("Number of respondents who saw each movie")
plt.xlabel("Episode")
plt.ylabel("Respondents")
plt.show()

**Episode V** is also the most viewed movie (758 respondents), followed by Episode VI (738). Among the prequels, Episode I was seen the most (673), and Episode III the least (550).

The number of views roughly follows the rankings: the original trilogy movies were seen by more respondents and are ranked higher. A possible explanation is that the original movies have been out for much longer, and people who liked them were more likely to watch them again or recommend them. Episode I is an exception — it was seen by many people (it was the long-awaited first new movie in 16 years) but is ranked poorly.

## Exploring the Data by Binary Segments

We split the respondents by gender and compare the average rankings and the number of views for each movie.

In [ ]:
males = star_wars[star_wars["Gender"] == "Male"]
females = star_wars[star_wars["Gender"] == "Female"]
print(len(males), len(females))

comparison = pd.DataFrame({
    "male_ranking": males[ranking_cols].mean().values,
    "female_ranking": females[ranking_cols].mean().values,
    "male_seen_pct": (males[seen_cols].mean() * 100).values,
    "female_seen_pct": (females[seen_cols].mean() * 100).values
}, index=episodes)
print(comparison.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison[["male_ranking", "female_ranking"]].plot.bar(ax=axes[0], rot=0, title="Average ranking by gender (lower is better)")
comparison[["male_seen_pct", "female_seen_pct"]].plot.bar(ax=axes[1], rot=0, title="% of respondents who saw each movie")
plt.show()

There are 497 male and 549 female respondents.

- **Both genders rank Episode V as the best movie** (2.46 for men and 2.57 for women).
- Men rank the prequels lower than women do: Episode I gets 4.04 from men but 3.43 from women, and Episode II gets 4.22 vs. 3.95. Men prefer Episode IV more (3.00 vs. 3.54).
- A much larger share of men than women saw each movie — e.g. 78.9% of men vs. 64.3% of women saw Episode V, and 72.6% vs. 54.3% saw Episode I.

So men in the sample watched Star Wars more, and they favor the original trilogy more strongly than women do.

We also compare respondents who consider themselves Star Wars fans with those who don't.

In [ ]:
fan_col = "Do you consider yourself to be a fan of the Star Wars film franchise?"
fans = star_wars[star_wars[fan_col] == True]
non_fans = star_wars[star_wars[fan_col] == False]
print(len(fans), len(non_fans))

fan_comparison = pd.DataFrame({
    "fan_ranking": fans[ranking_cols].mean().values,
    "non_fan_ranking": non_fans[ranking_cols].mean().values
}, index=episodes)
print(fan_comparison.round(2))

fan_comparison.plot.bar(rot=0, title="Average ranking: fans vs. non-fans (lower is better)")
plt.show()

552 respondents consider themselves fans, and 284 don't (the rest didn't answer).

- **Fans** prefer the original trilogy by a wide margin: Episode V (2.33), Episode VI (2.83) and Episode IV (2.93) are their top three, while the prequels get rankings above 4.
- **Non-fans** rank Episode V the highest too (2.86), but surprisingly their second favorite is **Episode I** (2.94), and they rank Episode IV much lower (3.93) than fans do.

The fact that The Empire Strikes Back comes first in both groups makes the conclusion robust, while the preference for the original trilogy is mostly driven by fans.

## Conclusion

- *The Empire Strikes Back* (Episode V) is the **highest-ranked** movie (average ranking 2.51) and the **most viewed** one (758 respondents). It's the favorite of both men and women, and of fans and non-fans alike — so the rest of America does seem to agree that it's the best of the bunch.
- The original trilogy (IV–VI) is ranked better than the prequels (I–III), and this preference is strongest among men and self-declared fans.
- Men in the sample saw every movie more often than women.

Possible next steps: segment the data by other columns, such as `Education`, `Location (Census Region)` or `Which character shot first?`, and clean the character favorability columns (columns 15–28) to find the most liked and most disliked characters.